## Imports & Download

In [ ]:
import kagglehub
import os

print("Downloading GTSRB Dataset...")
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")

# Define the data path for the rest of the notebook
data_path = os.path.join(path, 'Train') 
if not os.path.exists(data_path):
    data_path = os.path.join(path, 'train') # Fallback for lowercase

print(f"Dataset downloaded and the train folder located at: {data_path}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

try:
    from CNN_model import TrafficSignCNN
    print("Successfully imported TrafficSignCNN from CNN_model.py")
except ImportError:
    print("ERROR: Could not find CNN_model.py. Please upload it to Colab.")

# Force PyTorch to use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Data Preparation

In [ ]:
# 1. Define Data Augmentation & Normalization for Training
train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),           # Augmentation: Slight rotations
    transforms.ColorJitter(brightness=0.2),  # Augmentation: Lighting variations
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))     # Standardize pixel values
])

test_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2. Load the full dataset using the path from Cell 1
print(f"Loading images from {data_path}...")
full_dataset = datasets.ImageFolder(root=data_path, transform=train_transforms)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = test_transforms

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Loaded {len(train_dataset)} training images and {len(val_dataset)} validation images.")

## Training

In [ ]:
# --- CELL 4: TRAINING ---

model = TrafficSignCNN(num_classes=43).to(device)
criterion = nn.CrossEntropyLoss()

# Improvement: L2 Regularization (weight_decay) added to the Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 

epochs = 10
print(f"Starting Training for {epochs} epochs on {device}...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

print("Training Complete!")

## Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []

print("Evaluating model on validation set...")
with torch.no_grad()
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"\\n Validation Accuracy: {acc * 100:.2f}%")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(16, 12))
sns.heatmap(cm, annot=False, cmap='Blues', fmt='d')
plt.title("Traffic Sign CNN Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Sign Class", fontsize=12)
plt.ylabel("True Sign Class", fontsize=12)
plt.show()